In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import zarr
import pandas as pd
import polars as pl
import numpy as np
from plotnine import *

from anngeno import AnnGeno
from scripts import get_burdens, get_burdens_faster, get_correlation

## Quick checks

In [ ]:
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/genebass_genes_161k_wes.zarr'
root = zarr.group(zarr_burdens_path)
(np.isnan(root['sum_burdens'][:, 1])==False).sum()

In [ ]:
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/temp_gb.zarr'
root = zarr.group(zarr_burdens_path)
(np.isnan(root['sum_burdens'][:, 1])==False).sum()

## Debug

In [ ]:
import sys
from tqdm import tqdm

config_path = config_path
output_zarr = zarr_burdens_path
anno_scores_path = None
max_burden = True
only_snps = False
overwrite = True
debug = True

In [ ]:
with open(config_path) as f:
    config = yaml.safe_load(f)

associations_df_path = config.get("associations_df_path")
all_annotation_list = []

if anno_scores_path:
    anno_scores_df = pl.read_parquet(anno_scores_path)
    available_annotations = list(set(anno_scores_df.columns) - set(["chrom", "pos", "ref", "alt", "region"]))
    all_annotation_list.extend(available_annotations)
    
else:
    rare_variant_annotations_dict = config.get('rare_variant_annotations')
    if rare_variant_annotations_dict:
        for category in rare_variant_annotations_dict.values():
            all_annotation_list.extend(category)

zarr_file_path = output_zarr
anno_chunk_size = 1


# Function get_burdens_array():
print(f"Zarr file does not exist at {zarr_file_path}, creating a new one.")
get_burdens_kwargs = {
    "config": config,
    "associations_df_path": associations_df_path,
    "annotation_list": all_annotation_list,
    "max_burden": max_burden,
    "only_snps": only_snps,
    "debug": debug,
}
if anno_scores_path:
    get_burdens_kwargs["new_anno_df"] = anno_scores_df

# gene_burdens_sum_df, gene_burdens_max_df, sample_id_arr, gene_id_list = get_burdens.get_burdens_array(**get_burdens_kwargs)

maf = config.get("association_testing_maf")

associations_df = pl.read_parquet(associations_df_path)
if debug:
    print("Debug is True, using only 5 associations")
    associations_df = associations_df.head()
genes = associations_df['gene'].unique()

print("Loading AnnGeno file")
anngeno_file = config.get("anngeno_file")
ag = AnnGeno(filename=anngeno_file, filemode="r")

print(f"Filtering for variants with MAF < {maf}")

# --- MODIFIED LINE START ---
# Convert Polars Series to a Python list for direct use in isin() with pandas
genes_list = genes.to_list()
variants_to_keep_df = ag.annotations[
    (ag.annotations['MAF'] < maf) & (ag.annotations['region'].isin(genes_list))
]
variants_to_keep = set(variants_to_keep_df["id"])
variants_to_keep_df


In [ ]:
all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

print(all_annotation_list, ' annotations')

gene_id_list = list(genes)
gene_burdens_sum = []
gene_burdens_max = []
print(f"Starting to compute gene burdens for {len(genes)} genes")
for gene in tqdm(gene_id_list):
    gis_sum, gis_max = get_burdens.get_gene_burdens(ag, gene, all_annotation_list, max_burden)
    gene_burdens_sum.append(gis_sum)
    if max_burden:
        gene_burdens_max.append(gis_max)

gene_burdens_sum_df = np.stack(gene_burdens_sum, axis=1)  # (n_samples, n_genes, n_annotations)
if max_burden:
    gene_burdens_max_df = np.stack(gene_burdens_max, axis=1)  # (n_samples, n_genes, n_annotations)
    print("Returning sum and max burden.")
    gene_burdens_sum_df, gene_burdens_max_df, ag.samples, gene_id_list

gene_burdens_sum_df

In [ ]:
(np.isnan(gene_burdens_sum_df)==False).sum()

In [ ]:
with open(config_path) as f:
    config = yaml.safe_load(f)

associations_df_path = config.get("associations_df_path")
all_annotation_list = []

if anno_scores_path:
    anno_scores_df = pl.read_parquet(anno_scores_path)
    available_annotations = list(set(anno_scores_df.columns) - set(["chrom", "pos", "ref", "alt", "region"]))
    all_annotation_list.extend(available_annotations)
else:
    rare_variant_annotations_dict = config.get('rare_variant_annotations')
    if rare_variant_annotations_dict:
        for category in rare_variant_annotations_dict.values():
            all_annotation_list.extend(category)

zarr_file_path = output_zarr
anno_chunk_size = 1

print(f"Zarr file does not exist at {zarr_file_path}, creating a new one.")
get_burdens_kwargs = {
    "config": config,
    "associations_df_path": associations_df_path,
    "annotation_list": all_annotation_list,
    "max_burden": max_burden,
    "only_snps": only_snps,
    "debug": debug,
}
if anno_scores_path:
    get_burdens_kwargs["new_anno_df"] = anno_scores_df

gene_burdens_sum_df, gene_burdens_max_df, sample_id_arr, gene_id_list = get_burdens.get_burdens_array(**get_burdens_kwargs)

gene_burdens_sum_df


## Test new code

In [ ]:
config_path = './config_genebass.yaml'

zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/temp_gb.zarr'

In [ ]:
get_burdens_faster.compute_and_store_burdens(
    config_path=config_path,
    output_zarr=zarr_burdens_path,
    max_burden=True,
    only_snps=False,
    # overwrite=True,
    # debug=True,
)

In [ ]:
root = zarr.group(zarr_burdens_path)
(np.isnan(root['sum_burdens'][:, 1])==False).sum()

In [ ]:
with open(config_path) as f:
    config = yaml.safe_load(f)
associations_df_path = config.get("associations_df_path")

maf = config.get("association_testing_maf")

associations_df = pl.read_parquet(associations_df_path)
genes = associations_df['gene'].unique()

print("Loading AnnGeno file")
anngeno_file = config.get("anngeno_file")
ag = AnnGeno(filename=anngeno_file, filemode="r", low_mem=True)
# ag = AnnGeno(filename=anngeno_file, filemode="r")

print(f"Filtering for variants with MAF < {maf}")

# Convert Polars Series to a Python list for direct use in isin() with pandas
genes_list = genes.to_list()

ag.all_variant_count

In [ ]:
variants_to_keep_df = ag.annotations.filter((pl.col('MAF') < maf) & (pl.col('region').is_in(genes_list)))

variants_to_keep_df

ag.subset_variants(variants_to_keep_df.select(pl.col("id")))
# # ag.annotations.select(pl.len())
# ag.all_variant_count

In [ ]:
snp_variants = ag.annotations.filter(
    (pl.col("ref").str.len_chars() == 1) & (pl.col("alt").str.len_chars() == 1)
)

ag.subset_variants(snp_variants.select(pl.col('id')))

ag.annotations.select(pl.len())

## Compute burdens

In [ ]:
# Compute burdens
get_burdens.compute_burdens(config_path, zarr_burdens_path, max_burden=False, only_snps=True)

In [ ]:
anno_scores_path = '/s/project/deeprvat/ukb_gym/new_annotations/flashzoi_snp_scores.parquet'
# anno_scores_path = '/s/project/deeprvat/ukb_gym/new_annotations/flashzoi_ag_snp_scores.parquet'
# anno_scores_path = '/s/project/deeprvat/ukb_gym/new_annotations/absplice2_pangolin.parquet'

get_burdens.add_new_anno_burdens(config_path, zarr_burdens_path, anno_scores_path, max_burden=False)

## Compute correlations

In [ ]:
config_path = './config.yaml'
# zarr_file_path = '/s/project/deeprvat/ukb_gym/burdens/burdens_onlysum_onlysnps_no_vars_na.zarr'
zarr_file_path = '/s/project/deeprvat/ukb_gym/burdens/temp.zarr'

root = zarr.group(zarr_file_path)
zarr_annotations = root["annotations"]
zarr_annotations[:]

In [ ]:
root['sum_burdens'][:, 1]

In [ ]:
rho_df = get_correlation.compute_correlations(config_path, zarr_file_path, max_burden=False)

rho_df

In [ ]:
rank_corr_df = rho_df
# rank_corr_df = pd.read_parquet('/s/project/deeprvat/ukb_gym/data/spearman_rho/correlation_all_no_vars_na.parquet')
# rank_corr_df = rank_corr_df[~rank_corr_df['annotation'].isin(annotations_to_exclude)]
config_path = './config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df['category'] = pd.Categorical(rank_corr_df['category'], categories=['plof', 'missense', 'splicing', 'regulatory', 'misc'], ordered=True)
rank_corr_df

In [ ]:
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['spearman_correlation'])
rank_corr_df['median_abs_correlation'] = rank_corr_df.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_df = rank_corr_df.sort_values('median_abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

rank_corr_df = rank_corr_df[~rank_corr_df['category'].isna()]
(
    ggplot(rank_corr_df.query("aggregation == 'sum'"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 10),
    )
)

In [ ]:
(
    ggplot(rank_corr_df.query("(aggregation == 'sum') & (category == 'regulatory')"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 7),
    )
)

## AbSplice2 debug

In [ ]:
a2_ukbg = pd.read_parquet('/s/project/deeprvat/ukb_gym/new_annotations/absplice2_pangolin.parquet')
a2_ukbg

In [ ]:
subset_rcdf = rank_corr_df[rank_corr_df.gene.isin(a2_ukbg.region.astype(str).unique())]
subset_rcdf

In [ ]:
(
    ggplot(rank_corr_df.query("(aggregation == 'sum') & (category == 'splicing')"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 7),
    )
)

# Phenotype vs GIS plot

In [ ]:
import zarr
from scripts import get_correlation

def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = get_correlation.cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation

In [ ]:
sorted(root.attrs)

In [ ]:
phenotype = 'LDL_direct_statin_corrected' #'Urate'
gene_num = '9138' #'9231'
annotation = 'CADD_raw' #'pangolin_score'
burden_type = 'sum'

plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_file_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"{annotation} - {round(c, 4)}")

)